# CORES: an AI assistant for government forms

**Closed and Open Repository Enhanced System.** MSc Artificial Intelligence dissertation, University of Stirling (2024). Author: Adeshina Ayodele.

This notebook is the original research prototype, cleaned for publication:

* All outputs have been cleared.
* The original form documents are **not included**. They belong to their publisher. Add your own blank form templates to `documents/` to run it.
* No API keys are stored here. Set `OPENAI_API_KEY` in Colab Secrets or a local `.env` file (see `.env.example`).
* No real personal data is included.

See `README.md` for the architecture, evaluation results and limitations.


# **Background**
The field of Generative AI Assistants for Government forms has been a subject of growing interest and scholarly investigation in recent years. As a need to address the challenge of assisting individuals, particularly those vulnerable, in completing government forms like the Adult Disability Payment Form. Understanding the various perspectives and developments within this domain has become increasingly crucial.


## Problem Statement
Develop an AI Assistant Chatbot in filling Government Disability Forms

In [ ]:
# pip install all the necessary dependencies
!pip install python-dotenv
!pip install requests
!pip install langchain==0.1.0
!pip install streamlit
!pip install openai
!pip install pdfplumber
!pip install PyMuPDF  # PyMuPDF
!pip install pyngrok
!pip install tiktoken
!pip install faiss-gpu
!pip install faiss-cpu
!pip install python-docx python-pptx
!apt-get install -y tesseract-ocr
!pip install pytesseract pillow docx2txt

### **Core Libraries**

*openai:* The official OpenAI Python library, used to interact with OpenAI's models like GPT-3.5 and GPT-4 for text generation, translation, question answering, and more.

*requests:* A popular library for making HTTP requests to web services and APIs, allowing you to fetch data from the internet.

*os:* Python's built-in library for interacting with the operating system, providing functionalities like file system navigation and environment variable management.

*datetime:* Python's built-in library for handling dates and times, useful for scheduling tasks, logging events, and working with time-sensitive data.

*pdfplumber*: This is a Python library for extracting text and other information from PDF files.

*PyMuPDF (fitz)*: This is a Python binding for the MuPDF library, which is used for working with PDF and other document formats.

*pyngrok*: This is a Python library that allows you to expose a local web server to the internet using the ngrok service.

*tiktoken*: This is a Python library for fast encoding and decoding of text using the Byte Pair Encoding (BPE) algorithm.

*faiss-gpu and faiss-cpu*: These are Python libraries for efficient similarity search and clustering of dense vectors, with GPU and CPU support respectively.

*python-docx and python-pptx*: These are Python libraries for creating and manipulating Microsoft Word (DOCX) and PowerPoint (PPTX) documents.

*tesseract-ocr*: This is an open-source Optical Character Recognition (OCR) engine that can be used to extract text from images.

*pytesseract*: This is a Python wrapper for Google's Tesseract-OCR Engine, which allows you to use Tesseract-OCR from within your Python code.

*pillow*: This is a Python Imaging Library (PIL) that provides image processing capabilities.

*docx2txt*: This is a Python library that can extract text from Microsoft Word (DOCX) documents.

**LangChain**

*langchain.chat_models.ChatOpenAI*: Provides a wrapper for interacting with OpenAI's chat models in a more structured way, facilitating the creation of conversational agents.

*langchain.chains.LLMChain:* A core component of LangChain that allows you to chain together language models and other components to create more complex workflows.

*langchain.memory.ConversationBufferWindowMemory*: A type of memory that stores a history of the conversation, allowing the language model to maintain context and provide more relevant responses.

*langchain.prompts.PromptTemplate:* A tool for creating and managing prompts for language models, making it easier to structure and customize your interactions with the model.
Other

*streamlit:* A framework for building interactive web applications for data science and machine learning, allowing you to easily create user interfaces for your models and data.

*python-dotenv:* A library for loading environment variables from a .env file, helping you manage sensitive information like API keys securely.

In [ ]:
import os
import openai

# Load the OpenAI API key without ever writing it into the notebook.
# In Google Colab it comes from Colab Secrets; elsewhere from a local .env file.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()


In [ ]:
# This code gets the API key from the environment variable
openai.api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# Original research setup: the forms were stored on Google Drive.
# To run locally, place your own form templates in the documents/ folder instead.
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)


In [ ]:
# import the necessary libraries
import fitz  # PyMuPDF
import json
import requests
import time
import datetime
import warnings
import tiktoken
import faiss
import fitz  # PyMuPDF
from docx import Document
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.prompts import PromptTemplate
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import ConversationalRetrievalChain
from langchain.document_loaders import PyMuPDFLoader
from langchain.document_loaders import PyPDFLoader
from langchain.document_loaders import Docx2txtLoader
from langchain.schema import SystemMessage
from langchain.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    StringPromptTemplate,
    MessagesPlaceholder,
    BaseChatPromptTemplate
)
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.callbacks import get_openai_callback
import fitz  # PyMuPDF
from docx import Document
import docx2txt
import io
from PIL import Image
import pytesseract
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # This code reads local .env file

This is the LangChain Bot version with document access for Context Augmentation.
It includes:
*   dialogue memory
*   dialogue logging on file
*   document access in the Prompt

CAUTION: large pdf documents tend to slow down the ChatBot and cost more tokens. A reasonable size is up to 5 pages.







In [ ]:
# Optional (Colab only): upload form templates from your computer.
# from google.colab import files
# uploaded = files.upload()


1. from google.colab import files: This line imports the files module from the google.colab library. This module provides functions for interacting with the Colab file system, including uploading and downloading files.

2. uploaded = files.upload(): This line calls the upload() function from the files module. When executed, it displays a button in the Colab notebook that allows the user to select files from their local machine to upload. The uploaded files are stored in the Colab environment and a dictionary is returned, where keys are the file names and values are the file

In [ ]:
import os
import time
import datetime
import fitz  # PyMuPDF
import docx2txt
from PIL import Image
import pytesseract
from langchain.chat_models import ChatOpenAI
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferWindowMemory
from langchain.vectorstores import FAISS
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings

class DocumentProcessor:
    def __init__(self, chunk_size=1000, chunk_overlap=200):
        self.text_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.embeddings = OpenAIEmbeddings()
        self.vectorstore = None

    def process_pdf(self, pdf_path):
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()

            # This code extract images
            images = page.get_images(full=True)
            for img_index, img in enumerate(images):
                xref = img[0]
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]

                # This code uses PIL to open the images
                image = Image.open(io.BytesIO(image_bytes))

                # This code perform OCR on the image
                image_text = pytesseract.image_to_string(image)
                text += f"\nImage {img_index + 1} text:\n{image_text}\n"

        return text

    def process_docx(self, docx_path):
        # This code extract text
        text = docx2txt.process(docx_path)

        # This code extract images
        temp_dir = "temp_images"
        os.makedirs(temp_dir, exist_ok=True)
        docx2txt.process(docx_path, temp_dir)

        # This code Performs OCR on extracted images
        for img_file in os.listdir(temp_dir):
            if img_file.endswith(('png', 'jpg', 'jpeg', 'tiff')):
                img_path = os.path.join(temp_dir, img_file)
                image_text = self.extract_text_from_image(img_path)
                text += f"\nImage {img_file} text:\n{image_text}\n"
                os.remove(img_path)  #  This code Cleans up the temporary image file

        os.rmdir(temp_dir)  # This code removes the temporary directory
        return text

    def extract_text_from_image(self, image_path):
        image = Image.open(image_path)
        text = pytesseract.image_to_string(image)
        return text

    def process_file(self, file_path):
        _, file_extension = os.path.splitext(file_path)
        if file_extension.lower() == '.pdf':
            return self.process_pdf(file_path)
        elif file_extension.lower() in ['.docx', '.doc']:
            return self.process_docx(file_path)
        else:
            raise ValueError(f"Unsupported file type: {file_extension}")

    def add_document(self, file_path):
        text = self.process_file(file_path)
        texts = self.text_splitter.split_text(text)

        if self.vectorstore is None:
            self.vectorstore = FAISS.from_texts(texts, self.embeddings)
        else:
            self.vectorstore.add_texts(texts)

    def search(self, query, k=3):
        if self.vectorstore is None:
            return []
        return self.vectorstore.similarity_search(query, k=k)


In [ ]:
# Usage example:
processor = DocumentProcessor()
processor.add_document('documents/Referrral Form 03.pdf')
processor.add_document('documents/Referrral Form 02.pdf')
processor.add_document('documents/Referrral Form 01.pdf')
processor.add_document('documents/Nutritional Risk Assessment 01.pdf')
processor.add_document('documents/Nutritional Risk Assessment 02.pdf')
processor.add_document('documents/Nutritional Risk Assessment 03.pdf')
processor.add_document('documents/Nutritional Risk Assessment 04.pdf')
results = processor.search("personal details")
print(results)

In [ ]:
# Disability Form Assistant class
# is responsible for generating responses to user queries based on relevant document context,
# managing conversation history and user interactions,
# while saving dialogues for future reference.
class DisabilityFormAssistant:
    def __init__(self, pdf_paths, model="gpt-4", temperature=0.7, chunk_size=1000, chunk_overlap=200, k=3):
        self.api_key = os.getenv("OPENAI_API_KEY")
        openai.api_key = self.api_key
        self.model = model
        self.temperature = temperature
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.k = k
        self.llm = self._setup_llm()
        self.document_processor = DocumentProcessor(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        self.memory = ConversationBufferWindowMemory(memory_key="history", input_key="human_input")
        self.chat_chain = self._setup_chat_chain()

        for pdf_path in pdf_paths:
            self.add_document(pdf_path)

    def _setup_llm(self):
        return ChatOpenAI(model=self.model, temperature=self.temperature)

    def _setup_chat_chain(self):
        template = """
        Act as if you are a professional assistant specialized in helping people with disabilities fill out official documentation.\
        Your tone should be helpful, knowledgeable.\
        Your language style should be informative, friendly and culturally sensitive.\
        Your responses must be a very conversational and friendly style.\
        In your answer, write each sentence in one line to aid reading.\
        Your responses must be short, break to the next line after one sentence.\
        Your response should be in steps, ensure that each text field is filled before moving to the next.\
        Never ever mention that you are an AI or a language model.\
        You provide step-by-step guidance to fill out forms accurately.\
        Offer answers to questions about the entire process.\
        First welcome the user, Introduce yourself as an assistant with a random name of your choice. Keep and use this name alone within the session.\
        Be sensitive to the needs of users and their vulnerability by providing clear, simple, and supportive guidance.
        as shown in this example: "If you need any help understanding a term or section, just type 'explain' and I'll give you more information."\

        Obtain all procedure from the website https://www.mygov.scot/adult-disability-payment\
        Break down complex sections into smaller, manageable parts, ensuring the user can comfortably complete each step.\
        First welcome the user, Introduce yourself as an assistant with a random name of your choice. Keep and use this name alone within the session.\
        next ask what particular form the user requires assistant on.
        next follow the format of the uploaded forms that is exactly or most similar to particular form the user requires assistant on.
        unless user asks specific question(s)which you must answer first.\
        Use your capabilities to retrieve and provide relevant information dynamically based on user input as shown
        in the example: "I found some guidelines that explain this section in more detail. Would you like me to go over them with you?".\

        In your answer, write each sentence in one line to aid reading.\
        Your responses must be short, break to the next line after one sentence.\
        Include specific references to the name, sections and pages of the uploaded documents when applicable.\
        Encourage users to ask further questions or move on to the next section after completing a part of the form as shown
        in this example:  "You’ve made great progress! Would you like to review what we’ve done so far or continue to the next section?".\
        Provide , at the request of the user, a summary of the all the correct inputs the user has provided to fill out the form.\

        If you cannot find the answer in the documents or forms uploaded and on the website: https://www.mygov.scot/adult-disability-payment then ask the user to rephrase the query or question.\
        Whenever an error is encountered in this chat, provide helpful suggestions, clarify the user's intent, or offer alternative actions.\
        Gently prompt users to correct or complete any missing information as shown in the example: "Before we move on, let’s double-check this section to make sure everything is filled out correctly."\
        Whenever an enquiry or chat is out of scope of your objective, stay true to your objective: To offer assistance with filling for Government Forms.\
        Offer reassurance and acknowledge the user's effort as shown in this example: "You’re doing great! Remember, I’m here to help you every step of the way."\

        This is your prompt.

        Context from the document:
        {context}

        {history}
        Human: {human_input}
        Assistant:
        """
        prompt = PromptTemplate(input_variables=["context", "history", "human_input"], template=template)
        return LLMChain(llm=self.llm, prompt=prompt, memory=self.memory)

    def add_document(self, file_path):
        self.document_processor.add_document(file_path)

    def get_response(self, query):
        start_time = time.time()

        # This code retrieves relevant context from the vectorstore
        docs = self.document_processor.search(query, k=self.k)
        context = "\n".join([doc.page_content for doc in docs]) if docs else ""

        # This code generate response using the chat chain
        response = self.chat_chain.predict(human_input=query, context=context)

        end_time = time.time()
        response_time = end_time - start_time
        return response, response_time

    def get_user_rating(self):
        while True:
            try:
                rating = int(input("Please rate your experience (1-5): "))
                if 1 <= rating <= 5:
                    return rating
                else:
                    print("Please enter a number between 1 and 5.")
            except ValueError:
                print("Invalid input. Please enter a number.")

    def save_dialogue(self, dialogue):
        timestamp = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
        filename = f"Dialogue_{timestamp}.txt"
        with open(filename, "w") as file:
            file.write(dialogue)
        print(f"Dialogue saved to {filename}")


In [ ]:
def main():
    # creates an interactive loop to handle user inputs,
    # allows adding new documents, generates responses using the assistant,
    # logs the conversation, prompts for a user rating, and saves the dialogue to a file.
    pdf_paths = [
        "documents/Referrral Form 03.pdf",
        "documents/Referrral Form 02.pdf",
        "documents/Referrral Form 01.pdf",
        "documents/Nutritional Risk Assessment 01.pdf",
        "documents/Nutritional Risk Assessment 02.pdf",
        "documents/Nutritional Risk Assessment 03.pdf"
    ]  # PDF paths are added here
    assistant = DisabilityFormAssistant(pdf_paths)
    dialogue = ""

    print("Welcome!This system is designed to assist with filling Government forms.")
    print("We are here to help you fill out your government forms, guiding you every step of the way.")
    print("If anything isn't clear or if you need an explanation, just ask—no question is too small.")
    print("We can go at your own pace, and We're happy to repeat or clarify as needed. Let's get started whenever you're ready!.")
    print()
    print("Please note:")
    print("You may type 'exit' to end the conversation at any point.")
    print("You may also type 'add document' to add a new document to the knowledge base if you are authorized to do so.")
    print()
    print("Please type your first query and an assistant is automatically assigned to you.")
    while True:
        user_input = input("User > ")
        dialogue += f"User: {user_input}\n"

        if user_input.lower() == 'exit':
            break
        elif user_input.lower() == 'add document':
            file_path = input("Enter the path to the document: ")
            assistant.add_document(file_path)
            print("Document added successfully!")
            dialogue += f"Document added: {file_path}\n"
            continue

        response, response_time = assistant.get_response(user_input)
        print(f"Assistant: {response}")
        print(f"Response time: {response_time:.2f} seconds")

        dialogue += f"Assistant: {response}\n"
        dialogue += f"Response time: {response_time:.2f} seconds\n\n"

    rating = assistant.get_user_rating()
    dialogue += f"User rating: {rating}/5\n"

    assistant.save_dialogue(dialogue)
    print(f"Thank you for your rating: {rating}/5")
    print("Thank you for using the Disability Form Assistant. Goodbye!")

if __name__ == "__main__":
    main()

# Break down:
1. I have added the necessary libraries for PDF processing and vector storage.

2. The process_pdf function extracts text from the PDF, splits it into chunks, and creates a vector store for efficient searching.

3. The DisabilityFormAssistant class now takes a pdf_path parameter and uses the processed PDF as the primary source of information.

4. The get_response method now retrieves relevant context from the vector store before generating a response.
The prompt template has been updated to include the context from the PDF.